# Colab pipeline — chạy lại theo seed với SHAP trên **100% tập test** (Sparkov và ULB)

Notebook này nối thẳng Google Colab (GPU) vào repo để chạy lại **một dataset mỗi lần** (`DATASET = "sparkov"` hoặc `"ulb"` ở ô 1)
theo đúng thiết kế chính thức (10 seed, toàn bộ train, 5 mô hình × 5 kỹ thuật imbalance), với SHAP tính trên **toàn bộ tập test**:
Sparkov 555.719 dòng, ULB 56.745 dòng (ULB đã **bỏ 1.081 dòng trùng khớp** rồi chia theo thời gian — `scripts/prepare_ulb.py`).
PR-AUC luôn tính trên toàn bộ test. Mỗi dataset có thư mục kết quả và cache riêng (ULB nằm trong thư mục con `ulb/`), nên chạy xong Sparkov rồi đổi
`DATASET = "ulb"` và chạy lại từ ô 1 không ghi đè gì.

**Trước khi chạy**
1. `Runtime → Change runtime type → A100 GPU` + **High-RAM**. RAM hệ thống và số lõi CPU mới là thứ cần nhiều (CatBoost SHAP chính xác chạy trên CPU).
2. Cột trái → 🔑 **Secrets**, bật "Notebook access" cho:
   - `KAGGLE_USERNAME`, `KAGGLE_KEY` (KHÔNG BẮT BUỘC): chỉ cần nếu muốn tải dữ liệu thô từ Kaggle. Sparkov: ném `train_encoded.parquet` và `test_encoded.parquet` vào `Drive/CIES_colab/data_processed/`.
     ULB: ném `creditcard.csv` (hoặc cặp `ulb_train.parquet` + `ulb_test.parquet` GỐC) vào `Drive/CIES_colab/data_processed/ulb_raw/`.
   - `GITHUB_TOKEN` — chỉ cần nếu repo là private.
3. **Tham số CatBoost mới** (tune CPU, depth ≤ 7, chọn bằng `scripts/select_catboost_params.py --write`): chưa push lên git thì đặt `best_params.json` /
   `best_params_ulb.json` vào `Drive/CIES_colab/params/`; ô 7 sẽ chép chúng vào bản clone và in ra tham số CatBoost đang dùng.
4. Chạy các ô **theo thứ tự**. Phần **Thiết lập (ô 1–8)** phải chạy lại sau mỗi lần kết nối lại; chúng không làm lại việc đã xong.

**Mọi thứ nặng đều nằm trên Google Drive** (`DRIVE_DIR`): dữ liệu đã tiền xử lý, cache resample, kết quả từng tổ hợp và log.
Mất kết nối / hết phiên → kết nối lại, chạy lại ô 1–8, rồi chạy lại đúng ô đang dở: tổ hợp nào đã xong thì bị bỏ qua.

> **Quy tắc thiết kế (xem `README.md`):** mỗi (mô hình, kỹ thuật) phải chạy trọn 10 seed trong **một** lần chạy, cùng thiết bị.
> Kết quả ghi vào `<RUN_NAME>_gpu.json` (XGBoost, ANN trên GPU), `<RUN_NAME>_rf.json` (Random Forest, SHAP trên GPU) và `<RUN_NAME>_cpu.json`
> (Logistic Regression, **CatBoost trên CPU + SHAP Regular**). CatBoost GPU/Approximate (Saabas) không phải Shapley chính xác nên KHÔNG dùng cho kết quả chính (`AGENTS.md`).
> Các file khác tên nên chạy song song không ghi đè nhau.


## Thiết lập (chạy lại sau mỗi lần kết nối lại)

In [ ]:
# ===== 1. CẤU HÌNH — chỉnh ở đây =====
REPO_URL    = "https://github.com/gnUrt1106/fraud-detection-CIES.git"
REPO_BRANCH = "feat/seed-rerun-stability"   # nhánh có chế độ chạy lại theo seed; đổi sang "main" nếu đã merge
DRIVE_DIR   = "/content/drive/MyDrive/CIES_colab"
RUN_NAME    = "colab_full"                   # kết quả: results/rerun/<RUN_NAME>_gpu.json và <RUN_NAME>_cpu.json

DATASET     = "sparkov"   # "sparkov" | "ulb" — chạy xong một dataset thì đổi giá trị này và chạy lại từ ô 1
assert DATASET in ("sparkov", "ulb")
SEEDS       = "0-9"
SHAP_ROWS   = "all"        # "all" = 100% tập test (đúng yêu cầu). Đừng đổi giữa chừng: mọi tổ hợp phải cùng một mẫu.
SHAP_BATCH  = 100_000       # số dòng mỗi đợt SHAP (chỉ ảnh hưởng RAM/tốc độ, không ảnh hưởng kết quả)

SMOTE_ENN_MODE  = "gpu"    # chỉ Sparkov: "gpu" = ENN trên GPU (nhanh) | "cpu" = chậm hàng giờ mỗi seed. ULB luôn dùng CPU (nhỏ, vài phút; script GPU chỉ dành cho Sparkov)
CATBOOST_ON_GPU = False    # False = CatBoost trên CPU + SHAP Regular (chính xác), nhất quán với ULB và các model khác. True = GPU + SHAP Approximate (Saabas): KHÔNG dùng cho kết quả chính
CACHE_WORKERS   = 6        # số tiến trình tạo cache SMOTE/ADASYN/Borderline song song (mỗi tiến trình ~1-2 GB RAM)
RUN_GROUPS      = ["gpu", "rf", "cpu"]   # nhóm train chạy trên runtime NÀY (tiết kiệm CCU): runtime GPU → ["gpu", "rf"] (XGBoost, ANN, Random Forest); runtime CPU-only → ["cpu"] (Hồi quy Logistic, CatBoost). Mặc định: cả ba trên một runtime.
PIN_VERSIONS    = True     # ghim sklearn/imblearn/xgboost/catboost/shap... theo requirements-lock.txt (cache resample phụ thuộc phiên bản)

In [ ]:
# ===== 2. Kiểm tra máy =====
import os, sys, subprocess, shutil, json, time
from pathlib import Path

def sh(cmd, cwd=None, check=True):
    """Chạy lệnh shell, in output theo thời gian thực (subprocess không tự hiện output trong ô Colab)."""
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    rc = p.wait()
    if check and rc != 0:
        raise RuntimeError(f"Lệnh lỗi (mã {rc}): {cmd}")
    return rc

print("Python:", sys.version.split()[0], "(requirements-lock.txt tạo trên 3.12)")
gpu = subprocess.run("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout.strip()
HAS_GPU = bool(gpu)
print("GPU:", gpu or "KHÔNG CÓ — chỉ chạy được các bước CPU (cache, CatBoost, Logistic Regression)")
import psutil
print(f"RAM: {psutil.virtual_memory().total / 2**30:.1f} GB | CPU: {os.cpu_count()} lõi | "
      f"đĩa trống /content: {shutil.disk_usage('/content').free / 2**30:.0f} GB")

In [ ]:
# ===== 3. Gắn Google Drive =====
from google.colab import drive
drive.mount("/content/drive")

DRIVE = Path(DRIVE_DIR)
for d in ("data_processed", "resampled_rerun", "results_rerun", "logs", "params"):
    (DRIVE / d).mkdir(parents=True, exist_ok=True)
print("Thư mục Drive:", DRIVE)

In [ ]:
# ===== 4. Lấy mã nguồn từ GitHub =====
REPO = Path("/content/fraud-detection-CIES")
url = REPO_URL
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    if token:
        url = REPO_URL.replace("https://", f"https://{token}@")
except Exception:
    pass  # repo public thì không cần token

if not REPO.exists():
    sh(f"git clone --branch {REPO_BRANCH} {url} {REPO}")
else:
    sh(f"git -C {REPO} fetch origin {REPO_BRANCH} && git -C {REPO} checkout {REPO_BRANCH} && git -C {REPO} pull --ff-only origin {REPO_BRANCH}")
sh(f"git -C {REPO} log -1 --format='commit %h %s'")
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

In [ ]:
# ===== 5. Cài thư viện (giữ nguyên torch/numpy/pandas của Colab) =====
# Chỉ ghim các thư viện quyết định kết quả resample/model/SHAP. Nếu pip đòi khởi động lại:
# Runtime → Restart session, rồi chạy lại từ ô 1 (Drive và bản clone còn nguyên).
KEEP = {"scikit-learn", "imbalanced-learn", "xgboost", "catboost", "shap", "optuna", "kagglehub", "pyarrow", "python-dotenv"}
pins = []
for line in (REPO / "requirements-lock.txt").read_text().splitlines():
    line = line.strip()
    if "==" in line and not line.startswith("#") and line.split("==")[0].lower() in KEEP:
        pins.append(line)
names = sorted(KEEP)

if PIN_VERSIONS:
    rc = sh("pip install -q " + " ".join(f"'{p}'" for p in pins), check=False)
    if rc != 0:
        print("⚠️ Không cài được đúng phiên bản trong requirements-lock.txt → cài bản mới nhất (cache/kết quả có thể không trùng từng chữ số).")
        sh("pip install -q " + " ".join(names))
else:
    sh("pip install -q " + " ".join(names))

sh("pip list 2>/dev/null | grep -iE '^(scikit-learn|imbalanced-learn|xgboost|catboost|shap|optuna|torch|numpy|pandas|pyarrow) '")

In [ ]:
# ===== 6. Dữ liệu đã tiền xử lý (Sparkov: file encode; ULB: bỏ trùng + chia theo thời gian bằng scripts/prepare_ulb.py) =====
# File trên máy bạn: train_encoded.parquet (7.8MB) và test_encoded.parquet (3.2MB).
# Nếu đã ném vào Drive/MyDrive/CIES_colab/data_processed/ thì ô này tự nhận trong 1 giây!
# Nếu chưa có trên Drive, ô này sẽ mở hộp thoại Upload để bạn chọn 2 file từ máy (chỉ 11MB, không cần Kaggle).
PROC = REPO / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)
FILES = ["train_encoded.parquet", "test_encoded.parquet"]
drive_proc = DRIVE / "data_processed"
drive_proc.mkdir(parents=True, exist_ok=True)

if DATASET == "sparkov":
    missing = [f for f in FILES if not (drive_proc / f).exists()]

    if not missing:
        print("✅ Đã tìm thấy đầy đủ file encode trên Google Drive!")
        for f in FILES:
            shutil.copy2(drive_proc / f, PROC / f)
    else:
        print(f"Chưa có {missing} trên Drive.")
        print("👉 Bạn có 2 lựa chọn:")
        print("   1. Upload trực tiếp 2 file từ máy bạn (nhẹ ~11MB, nhanh nhất, không cần Kaggle)")
        print("   2. Tự tải lại từ Kaggle (~1.8GB thô)")
        ans = input("Bạn có muốn bấm upload 2 file parquet từ máy tính không? (y/n, mặc định y): ").strip().lower()
        if ans in ("", "y", "yes"):
            from google.colab import files as colab_files
            print("Vui lòng chọn 'train_encoded.parquet' và 'test_encoded.parquet' từ máy của bạn:")
            uploaded = colab_files.upload()
            for fname in uploaded:
                shutil.copy2(fname, drive_proc / fname)
                shutil.copy2(fname, PROC / fname)
                print(f"✅ Đã lưu {fname} vào Drive và workspace.")
        else:
            print("Tải dữ liệu thô từ Kaggle...")
            from google.colab import userdata
            for k in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
                os.environ[k] = userdata.get(k)
            sh("python -m src.data.download")

            import pandas as pd, joblib
            from src.config import RAW_DATA_DIR, TRAIN_FILE, TEST_FILE, SEED, TARGET_COL, ONEHOT_COLS, TARGET_ENCODE_COLS
            from src.data.preprocess import preprocess_sparkov
            from src.data.encoding import encode_train, encode_test

            raw_train = pd.read_csv(RAW_DATA_DIR / TRAIN_FILE)
            raw_test = pd.read_csv(RAW_DATA_DIR / TEST_FILE)
            train_df, test_df = preprocess_sparkov(raw_train), preprocess_sparkov(raw_test)
            enc_train, maps = encode_train(train_df, target_col=TARGET_COL, onehot_cols=ONEHOT_COLS,
                                           target_encode_cols=TARGET_ENCODE_COLS, n_splits=5, smoothing=10, random_state=SEED)
            enc_test = encode_test(test_df, maps, target_encode_cols=TARGET_ENCODE_COLS)
            assert list(enc_train.columns) == list(enc_test.columns)
            enc_train.to_parquet(PROC / "train_encoded.parquet", index=False)
            enc_test.to_parquet(PROC / "test_encoded.parquet", index=False)
            joblib.dump(maps, PROC / "encoding_maps.joblib")
            for f in FILES + ["encoding_maps.joblib"]:
                shutil.copy2(PROC / f, drive_proc / f)
            print("✅ Đã tiền xử lý xong và lưu lên Drive.")
else:
    drive_ulb, raw_dir = drive_proc / "ulb", drive_proc / "ulb_raw"
    drive_ulb.mkdir(exist_ok=True); raw_dir.mkdir(exist_ok=True)
    ULB_FILES = ["ulb_train.parquet", "ulb_test.parquet", "bao_cao_bo_trung.json"]
    if all((drive_ulb / f).exists() for f in ULB_FILES):
        print("✅ Đã có ULB đã bỏ trùng trên Drive (data_processed/ulb/).")
        for f in ULB_FILES:
            shutil.copy2(drive_ulb / f, PROC / f)
    else:
        has_pair = (raw_dir / "ulb_train.parquet").exists() and (raw_dir / "ulb_test.parquet").exists()
        if not (raw_dir / "creditcard.csv").exists() and not has_pair:
            print(f"Chưa có dữ liệu ULB gốc trong {raw_dir}.")
            ans = input("Upload creditcard.csv (~150 MB) từ máy? (y = upload, n = tải từ Kaggle; mặc định y): ").strip().lower()
            if ans in ("", "y", "yes"):
                from google.colab import files as colab_files
                for fname in colab_files.upload():
                    shutil.move(fname, raw_dir / fname)
            else:
                from google.colab import userdata
                import kagglehub
                from src.config import KAGGLE_DATASET_ULB, ULB_FILE
                for k in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
                    os.environ[k] = userdata.get(k)
                shutil.copy2(Path(kagglehub.dataset_download(KAGGLE_DATASET_ULB)) / ULB_FILE, raw_dir / "creditcard.csv")
        sh(f"python scripts/prepare_ulb.py --raw {raw_dir} --out {PROC} --force")   # bỏ dòng trùng, chia 80/20 theo Time
        for f in ULB_FILES:
            shutil.copy2(PROC / f, drive_ulb / f)
        print("✅ Đã bỏ trùng và lưu ULB lên Drive (data_processed/ulb/).")

# --- Kiểm tra dữ liệu ---
from src.explainability import rerun_stability as rs
X_tr, y_tr, X_te, y_te, _, _ = rs.load_encoded_data(PROC, DATASET)
N_TEST, N_FRAUD_TEST = len(y_te), int(y_te.sum())
fp = rs.data_fingerprint(X_tr, y_tr, X_te, y_te)
print(f"\n📊 Dữ liệu: Train {len(y_tr):,} dòng | Test {N_TEST:,} dòng ({N_FRAUD_TEST:,} fraud)")
print(f"🔑 Dấu vân tay dữ liệu: {fp}")
ref_file = REPO / "results" / "rerun" / "main.json"
if DATASET == "sparkov" and ref_file.exists():
    ref = json.load(open(ref_file))["results"][0]["config"]["data_fingerprint"]
    print("✅ Trùng khớp 100% với dữ liệu của các thí nghiệm trước!" if ref == fp else "⚠️ Khác dấu vân tay cũ.")
if DATASET == "ulb":
    assert (len(y_tr), int(y_tr.sum()), N_TEST, N_FRAUD_TEST) == (226_981, 399, 56_745, 74), (
        f"ULB không phải bản đã bỏ trùng (cần train 226.981/399 fraud, test 56.745/74; đang là {len(y_tr):,}/{int(y_tr.sum())}, "
        f"{N_TEST:,}/{N_FRAUD_TEST}). Xoá Drive/data_processed/ulb/ rồi chạy lại ô này.")
else:
    assert N_TEST == 555_719, f"Số dòng test bất thường: {N_TEST}"
del X_tr, y_tr, X_te, y_te


In [ ]:
# ===== 7. Trỏ kết quả và cache sang Drive =====
# Chỉ đụng vào bản clone trên Colab; repo gốc trên máy bạn và GitHub không bị ảnh hưởng.
RES = REPO / "results" / "rerun"
if not RES.is_symlink():
    if RES.exists():
        shutil.rmtree(RES)   # kết quả cũ (mẫu SHAP 20%) trong clone: bỏ ra để `merge` không trộn hai loại mẫu
    RES.symlink_to(DRIVE / "results_rerun")

CACHE = PROC / "resampled_rerun"
if not CACHE.is_symlink():
    if CACHE.exists():
        shutil.rmtree(CACHE)
    CACHE.symlink_to(DRIVE / "resampled_rerun")

# Thư mục của dataset đang chạy: Sparkov ở gốc, ULB LUÔN ở thư mục con ulb/ (run_rerun.py tự chọn đúng như vậy)
RES_D   = RES   if DATASET == "sparkov" else RES / "ulb"
CACHE_D = CACHE if DATASET == "sparkov" else CACHE / "ulb"
RES_D.mkdir(parents=True, exist_ok=True); CACHE_D.mkdir(parents=True, exist_ok=True)

# Tham số Optuna: nếu có bản mới trong Drive/params/ (vd. CatBoost depth ≤ 7) thì dùng bản đó thay cho bản trong git.
# Đã push bản mới lên nhánh thì xoá Drive/params/ để `git pull --ff-only` ở ô 4 không vướng thay đổi cục bộ.
BP_FILE = "best_params.json" if DATASET == "sparkov" else "best_params_ulb.json"
for name in ("best_params.json", "best_params_ulb.json"):
    src = DRIVE / "params" / name
    if src.exists():
        shutil.copy2(src, REPO / "results" / name)
        print(f"Dùng {name} từ Drive/params/")
try:
    _cb = json.load(open(REPO / "results" / BP_FILE))["catboost"]
    print("CatBoost sẽ chạy (CPU, SHAP Regular) với:", _cb["best_params"])
    print("   nguồn:", _cb.get("selection", {}).get("source_db", "KHÔNG có khối 'selection' — đây là tham số Optuna gốc"))
    if _cb["best_params"].get("depth", 0) > 7:
        print("⚠️ CatBoost depth > 7: SHAP chính xác trên 100% test sẽ rất chậm. Chạy scripts/tune_catboost.py + select_catboost_params.py --write rồi đặt file vào Drive/params/.")
except Exception as e:
    print("⚠️ Không đọc được tham số CatBoost:", e)

stale = DRIVE / "results_rerun" / "main.json"
if DATASET == "sparkov" and stale.exists():
    print("⚠️ Có main.json (kết quả mẫu 20% cũ) trong results_rerun trên Drive — chuyển ra ngoài trước khi `merge`, nếu không merge sẽ cảnh báo lệch mẫu SHAP.")
print("results/rerun →", os.readlink(RES))
print("data/processed/resampled_rerun →", os.readlink(CACHE))

In [ ]:
# ===== 8. Công cụ chạy nền (chạy tiếp được cả khi đóng tab; log lưu trên Drive) =====
from IPython.display import clear_output

JOBS = Path("/content/jobs"); JOBS.mkdir(exist_ok=True)
LOGS = DRIVE / "logs"
PROCS = {}

def _alive(name):
    if name in PROCS:
        return PROCS[name].poll() is None
    f = JOBS / f"{name}.pid"
    if not f.exists():
        return False
    try:
        os.kill(int(f.read_text()), 0)
        return True
    except OSError:
        return False

def launch(name, cmd):
    """Chạy `cmd` ở nền, log vào Drive/logs/<name>.log. Không chạy trùng nếu job cùng tên đang chạy."""
    if _alive(name):
        print(f"Job '{name}' đang chạy rồi — dùng watch('{name}') để xem.")
        return
    logf = open(LOGS / f"{name}.log", "a")
    logf.write(f"\n\n===== {time.strftime('%Y-%m-%d %H:%M:%S')} :: {cmd}\n"); logf.flush()
    p = subprocess.Popen(["bash", "-c", cmd], cwd=REPO, stdout=logf, stderr=subprocess.STDOUT, start_new_session=True)
    PROCS[name] = p
    (JOBS / f"{name}.pid").write_text(str(p.pid))
    print(f"Đã chạy '{name}' (pid {p.pid}). Xem: watch('{name}')")

def tail(name, n=25):
    f = LOGS / f"{name}.log"
    print("\n".join(f.read_text(errors="replace").splitlines()[-n:]) if f.exists() else "(chưa có log)")

def watch(name, every=30, n=25):
    """Theo dõi log tới khi job xong. Ngắt ô này (■) không dừng job."""
    try:
        while _alive(name):
            clear_output(wait=True)
            print(f"[{name}] đang chạy — {time.strftime('%H:%M:%S')}")
            tail(name, n)
            time.sleep(every)
    except KeyboardInterrupt:
        print("(đã ngừng theo dõi, job vẫn chạy)")
        return
    clear_output(wait=True)
    print(f"[{name}] ĐÃ KẾT THÚC")
    tail(name, n)

RR = "python scripts/run_rerun.py"
DSET = f"--dataset {DATASET}"
print("Sẵn sàng. Job đang chạy:", [n for n in list(PROCS) + [p.stem for p in JOBS.glob('*.pid')] if _alive(n)] or "không")

## Chạy

Thứ tự: **(9) trạng thái → (10) autopilot chạy hết → (14) gộp & kiểm tra → (15) tải kết quả**. Ô 9b (đo giờ) là tuỳ chọn.
Autopilot chạy song song cache, SMOTE-ENN và train; chỉ cần chạy lại ô 10 sau mỗi lần kết nối lại. Cache dùng chung cho mọi mô hình và nằm trên Drive.

In [ ]:
# ===== 9. Trạng thái (cache đã có, kết quả đã xong, SHAP trên bao nhiêu dòng) =====
sh(f"{RR} status --dataset {DATASET} --seeds {SEEDS}")

In [ ]:
# ===== 9b. (Tuỳ chọn) Đo giờ trước khi chạy thật — SHAP trên 100% test =====
launch(f"pilot_{DATASET}", f"{RR} pilot --dataset {DATASET} --models xgboost ann random_forest --shap-backend native --shap-rows {SHAP_ROWS} --name {RUN_NAME}")
watch(f"pilot_{DATASET}", every=15)


In [ ]:
# ===== 9c. (Tuỳ chọn) Đo giờ CatBoost CPU + SHAP Regular (chính xác) với tham số đang dùng =====
# Chạy TRƯỚC khi chạy thật: depth 11 (tham số Optuna gốc) sẽ cho số giờ rất lớn; depth ≤ 7 thì vừa phải.
launch(f"pilot_cb_{DATASET}", f"{RR} pilot --dataset {DATASET} --models catboost --force-cpu --catboost-shap-calc-type Regular --shap-rows {SHAP_ROWS} --name {RUN_NAME}")
watch(f"pilot_cb_{DATASET}", every=15)


In [ ]:
# ===== 9d. (Tuỳ chọn) Benchmark CatBoost CPU so với GPU: tốc độ, PR-AUC, SHAP, và GPU có tái lập từng bit không =====
# KHÔNG tạo kết quả chính (CatBoost kết quả chính = CPU + SHAP Regular). Dùng để có SỐ ĐO trả lời "sao không train CatBoost trên GPU?".
# Chỉ đưa "GPU không tái lập" vào hạn chế của khoá luận nếu dòng KẾT LUẬN ở cuối nói GPU không tái lập còn CPU thì có.
assert HAS_GPU, "Ô này cần runtime GPU"
sh(f"python scripts/bench_catboost_gpu.py --dataset {DATASET} --depth 7 --repeats 3")
shutil.copy2(REPO / "results" / "benchmarks" / f"catboost_gpu_vs_cpu_{DATASET}.json", DRIVE / "logs" / f"catboost_gpu_vs_cpu_{DATASET}.json")


In [ ]:
# ===== 9e. (Chỉ cần nếu còn CatBoost cũ trong <RUN_NAME>_gpu.json hoặc <RUN_NAME>_cpu.json) Chuyển chúng ra trước khi chạy CatBoost mới =====
# CatBoost cũ có thể nằm ở HAI nơi: _gpu.json (GPU + SHAP Approximate) và _cpu.json (CPU + SHAP Regular nhưng tham số depth cũ). Cả hai phải ra khỏi
# results/rerun, nếu không autopilot coi các tổ hợp đó là "đã xong" và bỏ qua => CatBoost sẽ lẫn hai bộ tham số.
# Mặc định chỉ IN những gì sẽ chuyển. Đặt APPLY_SUPERSEDE = True và chạy lại ô này để thật sự chuyển (có sao lưu trong Drive/results_superseded/).
APPLY_SUPERSEDE = False
for _grp in ("gpu", "cpu"):
    _f = RES_D / f"{RUN_NAME}_{_grp}.json"
    if not _f.exists():
        print(f"{_f.name}: không có file"); continue
    _n = sum(1 for r in json.load(open(_f)).get("results", []) if r.get("model_name") == "catboost")
    if _n == 0:
        print(f"{_f.name}: không có CatBoost"); continue
    print(f"\n--- {_f.name}: {_n} tổ hợp CatBoost")
    sh(f"python scripts/supersede_results.py --file {_f} --models catboost --dest-dir {DRIVE}/results_superseded" + (" --apply" if APPLY_SUPERSEDE else ""), check=False)


In [ ]:
# ===== 10. AUTOPILOT — chạy hết ở tốc độ tối đa, tự điều phối các job song song =====
# Một vòng lặp lo toàn bộ pipeline, mỗi khi đủ điều kiện thì tự bật job tiếp theo:
#   • cache SMOTE/ADASYN/Borderline: chia seed cho CACHE_WORKERS tiến trình
#   • SMOTE-ENN: CPU seed 0 (đối chứng) chạy song song với mọi thứ khác → GPU check → GPU cache các seed còn lại
#   • train 3 nhóm cùng lúc, mỗi nhóm lấy ngay các kỹ thuật đã có cache (chưa có SMOTE-ENN thì làm 4 kỹ thuật kia trước):
#       gpu = XGBoost + ANN (GPU)   rf = Random Forest (CPU + SHAP trên GPU)   cpu = Logistic Regression + CatBoost (CPU)
#   Tiết kiệm CCU: RUN_GROUPS ở ô 1 chọn nhóm nào chạy trên runtime này (GPU: ["gpu","rf"]; CPU-only: ["cpu"]). Hai runtime ghi hai file kết quả khác tên nên chạy song song được.
# Ngắt ô bằng ■ không dừng job; chạy lại ô này để tiếp tục điều phối. Tổ hợp đã xong luôn được bỏ qua.
from src.explainability.rerun_stability import parse_seeds

assert RUN_GROUPS and set(RUN_GROUPS) <= {"gpu", "rf", "cpu"}, "RUN_GROUPS chỉ gồm 'gpu', 'rf', 'cpu' (ô 1)"
assert HAS_GPU or set(RUN_GROUPS) == {"cpu"}, "Nhóm gpu/rf cần runtime GPU. Trên runtime CPU-only đặt RUN_GROUPS = ['cpu'] ở ô 1"
assert SHAP_ROWS == "all", "SHAP_ROWS phải là 'all' để dùng 100% test"
assert not CATBOOST_ON_GPU, "CatBoost GPU + SHAP Approximate (Saabas) không phải Shapley chính xác: không dùng cho kết quả chính (AGENTS.md). Đặt CATBOOST_ON_GPU = False."
SEED_LIST = parse_seeds(SEEDS)
ENN_MODE = SMOTE_ENN_MODE if DATASET == "sparkov" else "cpu"   # scripts/gpu_smote_enn.py chỉ dành cho Sparkov
if not HAS_GPU:
    ENN_MODE = "cpu"   # runtime CPU-only không có GPU để tạo cache SMOTE-ENN
if ENN_MODE == "gpu":
    assert 0 in SEED_LIST, "Seed 0 bắt buộc có mặt: bản CPU seed 0 là đối chứng của SMOTE-ENN GPU"
ALL_TECH = ["smote", "smote_enn", "adasyn", "borderline_smote", "class_weighting"]
BASIC = ["smote", "adasyn", "borderline_smote"]
for _grp in ("gpu", "cpu"):
    _g = RES_D / f"{RUN_NAME}_{_grp}.json"
    if _g.exists():
        _old = [r["combination"] for r in json.load(open(_g)).get("results", []) if r.get("model_name") == "catboost"]
        assert not _old, f"{_g.name} còn {len(_old)} tổ hợp CatBoost cũ: autopilot sẽ coi chúng là đã xong và bỏ qua (CatBoost sẽ lẫn hai bộ tham số). Chạy ô 9e để chuyển ra trước."
COMMON = f"--dataset {DATASET} --shap-rows {SHAP_ROWS} --shap-batch {SHAP_BATCH} --seeds {SEEDS}"

GROUPS_ALL = {"gpu": dict(models=["xgboost", "ann"], flags="--shap-backend native"),
          "rf":  dict(models=["random_forest"], flags="--shap-backend native"),
          "cpu": dict(models=["logistic_regression", "catboost"], flags="--force-cpu")}   # CatBoost: CPU + SHAP Regular (mặc định)
GROUPS = {k: v for k, v in GROUPS_ALL.items() if k in RUN_GROUPS}   # chỉ các nhóm của runtime này; TOTAL bên dưới chỉ tính các nhóm đó
TOTAL = {f"{m}__{t}" for g in GROUPS.values() for m in g["models"] for t in ALL_TECH}

def cache_file(t, s):
    return CACHE_D / f"seed{s:02d}" / f"train_encoded_{t}.parquet"      # ghi atomic: tồn tại = đã xong

def cache_ready(t):
    return t == "class_weighting" or all(cache_file(t, s).exists() for s in SEED_LIST)

if not HAS_GPU and DATASET == "sparkov":
    _thieu = [t for t in ALL_TECH if not cache_ready(t)]
    assert not _thieu, f"Runtime CPU-only không tự tạo cache Sparkov (SMOTE-ENN CPU mất hàng giờ mỗi seed và khác đường GPU đã dùng): thiếu cache {_thieu}. Tạo cache trên runtime GPU trước."

def done_combos():
    out = set()
    for f in RES_D.glob(f"{RUN_NAME}_*.json"):   # chỉ file của lần chạy này và của dataset này (không đếm main.json cũ)
        try:
            out |= {r["combination"] for r in json.load(open(f)).get("results", []) if r.get("complete")}
        except Exception:
            pass
    return out

TRIES = {}
def start(name, cmd, key=None, after=None, max_tries=3):
    """Bật job nếu chưa chạy; mỗi khối việc giống nhau chỉ thử lại tối đa max_tries lần (tránh lặp vô hạn khi lỗi thật)."""
    k = (name, key)
    if _alive(name) or (after and _alive(after)) or TRIES.get(k, 0) >= max_tries:
        return False
    TRIES[k] = TRIES.get(k, 0) + 1
    launch(name, cmd)
    return True

def autopilot(poll=90):
    while True:
        started = False
        done = done_combos()

        # (a) cache SMOTE / ADASYN / Borderline-SMOTE — chia seed cho nhiều tiến trình
        miss = [s for s in SEED_LIST if not all(cache_file(t, s).exists() for t in BASIC)]
        if miss and not any(_alive(f"cache_basic_{k}") for k in range(CACHE_WORKERS)):
            for k in range(CACHE_WORKERS):
                part = miss[k::CACHE_WORKERS]
                if part:
                    started |= start(f"cache_basic_{k}", f"{RR} cache --dataset {DATASET} --techniques {' '.join(BASIC)} --seeds {','.join(map(str, part))}",
                                     key=tuple(part))

        # (b) SMOTE-ENN chạy 100% bằng GPU (không tốn 2.5 tiếng chạy CPU)
        if ENN_MODE == "gpu":
            miss_enn = [s for s in SEED_LIST if not cache_file("smote_enn", s).exists()]
            if miss_enn:
                cmd_enn = (f"python scripts/gpu_smote_enn.py cache --seeds {','.join(map(str, miss_enn))} "
                           f"--skip-guard --q-chunk 4096 --db-chunk 524288")
                started |= start("enn_gpu", cmd_enn, key=tuple(miss_enn))
        else:
            miss_enn = [s for s in SEED_LIST if not cache_file("smote_enn", s).exists()]
            if miss_enn and not any(_alive(f"enn_cpu_{k}") for k in range(CACHE_WORKERS)):
                for k in range(CACHE_WORKERS):
                    part = miss_enn[k::CACHE_WORKERS]
                    if part:
                        started |= start(f"enn_cpu_{k}", f"{RR} cache --dataset {DATASET} --techniques smote_enn --seeds {','.join(map(str, part))}",
                                         key=tuple(part))

        # (c) train — mỗi nhóm lấy các kỹ thuật ĐÃ có đủ cache (không bao giờ tính resample dở trong lúc train)
        ready = [t for t in ALL_TECH if cache_ready(t)]
        for gname, g in GROUPS.items():
            todo = [t for t in ready if any(f"{m}__{t}" not in done for m in g["models"])]
            if todo:
                cmd = (f"{RR} train --models {' '.join(g['models'])} --techniques {' '.join(todo)} {g['flags']} "
                       f"{COMMON} --name {RUN_NAME}_{gname}")
                started |= start(f"train_{gname}", cmd, key=tuple(todo), max_tries=4)

        remaining = TOTAL - done
        alive = [n for n in {p.stem for p in JOBS.glob("*.pid")} | set(PROCS) if _alive(n)]
        clear_output(wait=True)
        n_cache = {t: sum(cache_file(t, s).exists() for s in SEED_LIST) for t in BASIC + ["smote_enn"]}
        print(f"{time.strftime('%H:%M:%S')} | tổ hợp xong {len(TOTAL) - len(remaining)}/{len(TOTAL)} | cache (số seed) {n_cache}")
        print("job đang chạy:", sorted(alive) or "không")
        for n in sorted(alive):
            print(f"--- {n}"); tail(n, 2)
        if not remaining:
            print("\n✅ XONG hết tổ hợp. Chạy ô 14b để gộp và kiểm tra."); return
        if not alive and not started:
            print("\n⛔ DỪNG: không còn job nào chạy mà vẫn thiếu tổ hợp (đã thử lại hết số lần cho phép).")
            print("Thiếu:", sorted(remaining)[:12], "..." if len(remaining) > 12 else "")
            print("Xem log trong Drive/CIES_colab/logs (đặc biệt enn_gpu.log nếu bước check SMOTE-ENN KHÁC bản CPU).")
            return
        try:
            time.sleep(poll)
        except KeyboardInterrupt:
            print("(đã ngừng điều phối, các job vẫn chạy)"); return

autopilot()

## Theo dõi, gộp, kiểm tra, tải kết quả

In [ ]:
# ===== 14a. Xem nhanh tiến độ bất cứ lúc nào =====
sh(f"{RR} status --dataset {DATASET} --seeds {SEEDS}")
for f in sorted(LOGS.glob("*.log")):
    j = f.stem
    print(f"\n--- {j} ({'đang chạy' if _alive(j) else 'không chạy'}) ---")
    tail(j, 4)

In [ ]:
# ===== 14a-gpu. Theo dõi GPU thời gian thực (Live GPU & Process Monitor) =====
# Ô này cập nhật liên tục mỗi 3 giây: VRAM, nhiệt độ, % GPU Util và các tiến trình CIES đang chạy.
# Ngắt ô (nút ■) bất cứ lúc nào, không ảnh hưởng đến các tiến trình đang chạy ngầm.
import time
import subprocess
from IPython.display import clear_output

def live_gpu_monitor(interval=3):
    try:
        while True:
            clear_output(wait=True)
            print("=" * 75)
            print(f"🖥️  LIVE GPU & TASK MONITOR — {time.strftime('%Y-%m-%d %H:%M:%S')}")
            print("=" * 75)
            
            # 1. Thông tin GPU
            res = subprocess.run(
                ["nvidia-smi", "--query-gpu=name,temperature.gpu,utilization.gpu,utilization.memory,memory.used,memory.total,power.draw",
                 "--format=csv,noheader,nounits"],
                capture_output=True, text=True
            )
            if res.returncode == 0 and res.stdout.strip():
                name, temp, gpu_u, mem_u, mem_used, mem_total, pwr = res.stdout.strip().split(", ")
                used_gb = float(mem_used) / 1024
                total_gb = float(mem_total) / 1024
                pct = (used_gb / total_gb) * 100
                bar = "█" * int(30 * pct / 100) + "░" * (30 - int(30 * pct / 100))
                print(f"Card     : {name}")
                print(f"Nhiệt độ : {temp}°C | Điện năng: {pwr} W")
                print(f"GPU Load : {gpu_u}% | Mem Load : {mem_u}%")
                print(f"VRAM     : [{bar}] {used_gb:.2f} / {total_gb:.1f} GiB ({pct:.1f}%)")
            else:
                print("Không tìm thấy GPU NVIDIA hoặc nvidia-smi!")

            # 2. Chi tiết VRAM các app đang chiếm
            print("\n" + "-" * 75)
            print("💾 APP ĐANG CHIẾM BỘ NHỚ GPU:")
            print("-" * 75)
            apps = subprocess.run(
                ["nvidia-smi", "--query-compute-apps=pid,process_name,used_memory", "--format=csv,noheader"],
                capture_output=True, text=True
            ).stdout.strip()
            print(apps if apps else "Không có app nào đang chiếm VRAM GPU")

            # 3. Tiến trình CIES đang chạy
            print("\n" + "-" * 75)
            print("⚙️  TIẾN TRÌNH CIES ĐANG CHẠY TRÊN HỆ THỐNG:")
            print("-" * 75)
            ps = subprocess.run(["ps", "-eo", "pid,%cpu,%mem,etime,cmd"], capture_output=True, text=True)
            cies_procs = [l for l in ps.stdout.splitlines() if any(k in l for k in ["run_rerun.py", "colab_run_all.sh", "catboost", "xgboost"])]
            if cies_procs:
                for p in cies_procs:
                    print("  " + p[:110])
            else:
                print("Chưa phát hiện tiến trình CIES nào đang chạy.")

            print("\n" + "=" * 75)
            print("Nhấn nút ■ (Stop) để dừng theo dõi.")
            time.sleep(interval)
    except KeyboardInterrupt:
        print("\nĐã dừng theo dõi GPU.")

live_gpu_monitor(interval=3)


In [ ]:
# ===== 14b. Gộp 25 tổ hợp và kiểm tra: đủ 25, đủ 10 seed, SHAP đúng 100% test =====
SUF = "" if DATASET == "sparkov" else "_ulb"
sh(f"{RR} merge --dataset {DATASET}")
S = json.load(open(REPO / "results" / f"cies_rerun_summary{SUF}.json"))
res, problems = S["results"], S["problems"]
bad_shap = [f"{r['model_name']}__{r['imbalance_technique']}" for r in res if r["config"].get("n_eval_shap") != N_TEST]
bad_seed = [f"{r['model_name']}__{r['imbalance_technique']}" for r in res if r["cies_metrics"].get("n_runs") != 10]
print(f"\nSố tổ hợp: {len(res)}/25 | cảnh báo của merge: {len(problems)}")
for p in problems:
    print("  ⚠️", p)
print(f"Tổ hợp SHAP KHÁC {N_TEST:,} dòng (phải rỗng): {bad_shap or 'không'}")
print(f"Tổ hợp chưa đủ 10 seed (phải rỗng): {bad_seed or 'không'}")
ok = len(res) == 25 and not problems and not bad_shap and not bad_seed
print("\n✅ ĐẠT — 25/25 tổ hợp, SHAP trên 100% test, đủ 10 seed." if ok else "\n❌ CHƯA ĐẠT — chạy tiếp/chạy lại các tổ hợp còn thiếu rồi gộp lại.")
import pandas as pd
display(pd.read_csv(REPO / "results" / f"cies_rerun_summary{SUF}.csv").pivot(index="model", columns="technique", values="cies").round(4))

In [ ]:
# ===== 15. Đóng gói để tải về máy (cũng nằm sẵn trên Drive) =====
SUF = "" if DATASET == "sparkov" else "_ulb"
OUT = Path("/content/export"); shutil.rmtree(OUT, ignore_errors=True)
OUT_RES = OUT / "results" / "rerun" / ("" if DATASET == "sparkov" else "ulb")
OUT_RES.mkdir(parents=True)
for f in RES_D.glob("*.json"):
    shutil.copy2(f, OUT_RES / f.name)
for f in (f"cies_rerun_summary{SUF}.json", f"cies_rerun_summary{SUF}.csv"):
    shutil.copy2(REPO / "results" / f, OUT / "results" / f)
zip_path = shutil.make_archive(str(DRIVE / f"KET_QUA_COLAB_FULL_SHAP{SUF}"), "zip", OUT)
print("Đã tạo:", zip_path)
from google.colab import files
files.download(zip_path)


## Đưa kết quả về repo trên máy

**Sparkov** (`KET_QUA_COLAB_FULL_SHAP.zip`)
1. Giải nén vào gốc repo `Fraud Detection - CIES`.
2. **Chuyển `results/rerun/main.json` (kết quả Sparkov cũ, mẫu SHAP 20%) ra thư mục riêng**, ví dụ `results/rerun_sample20_reference/`,
   rồi để `*_gpu.json` / `*_rf.json` / `*_cpu.json` trong `results/rerun/`. Nếu để lẫn, `python scripts/run_rerun.py merge` sẽ cảnh báo
   "Mẫu SHAP KHÁC nhau" và trùng tổ hợp. Năm mục CatBoost cũ (GPU/Approximate) trong `colab_full_gpu.json` chuyển sang `results/rerun_superseded/`.

**ULB** (`KET_QUA_COLAB_FULL_SHAP_ulb.zip`)
1. Giải nén vào gốc repo; kết quả nằm ở `results/rerun/ulb/`. Kết quả ULB cũ (chưa bỏ trùng / chạy trên máy khác) chuyển sang thư mục sao lưu trước.
2. Chép `ulb_train.parquet`, `ulb_test.parquet` đã bỏ trùng và `bao_cao_bo_trung.json` từ `Drive/CIES_colab/data_processed/ulb/` về `data/processed/`.

Cả hai: `python scripts/run_rerun.py merge --dataset sparkov|ulb` rồi `python scripts/plot_rerun.py` để cập nhật bảng và hình.
Sau đó cập nhật số liệu trong `README.md`, `reports/pipeline_report.md` và khóa luận (các số Sparkov hiện tại là của mẫu 20%).
